In [0]:
# generer les evenements 
from pyspark.sql.functions import (
    col,
    expr,
    floor,
    from_unixtime,
    lit,
    rand,
    when
)

incoming_path = (
    "/Volumes/retail_dev/ops/"
    "landing_volume/stock_events/incoming"
)

In [0]:
# Ingestion avec Auto Loader
from pyspark.sql.functions import col, current_timestamp
from pyspark.sql.types import (
    IntegerType,
    StringType,
    StructField,
    StructType,
    TimestampType
)

stock_event_schema = StructType([
    StructField("event_id", StringType(), False),
    StructField("stock_item_id", IntegerType(), False),
    StructField("store_id", IntegerType(), False),
    StructField("event_type", StringType(), False),
    StructField("quantity_change", IntegerType(), False),
    StructField("event_timestamp", TimestampType(), False),
    StructField("source_system", StringType(), False)
])

schema_path = (
    "/Volumes/retail_dev/ops/"
    "technical_volume/autoloader/stock_events/schema"
)

checkpoint_path = (
    "/Volumes/retail_dev/ops/"
    "technical_volume/checkpoints/stock_events"
)

stock_events_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", schema_path)
    .schema(stock_event_schema)
    .load(incoming_path)

    .withColumn("_ingested_at", current_timestamp())
    .withColumn("_source_file", col("_metadata.file_path"))
)

query = (
    stock_events_stream.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable("retail_dev.bronze.stock_events")
)

query.awaitTermination()

print("Ingestion Auto Loader terminée.")

In [0]:
# Verification 
stock_events_bronze = spark.table(
    "retail_dev.bronze.stock_events"
)

print(
    f"Nombre d'événements Bronze : "
    f"{stock_events_bronze.count()}"
)

display(
    stock_events_bronze
    .orderBy(col("event_timestamp").desc())
    .limit(20)
)